# Batch processing

hGRS is run image by image with the command-line tools `hgrs_enmap` and `hgrs_prisma` (see
[Usage](../usage.rst)). This notebook lists the Level-1 images of several sites, finds the CAMS file
covering each acquisition and writes the corresponding commands, to be run in a shell script or with
GNU `parallel`.

In [1]:
import os
import glob

opj = os.path.join

## CAMS files

The CAMS atmospheric composition forecasts are stored either as monthly files
(`cams_forecast_YYYY-MM.nc`) or as daily files (`YYYY/MM/DD/YYYY-MM-DD-cams-global-atmospheric-composition-forecasts.nc`).
The monthly file is used when it exists.

In [2]:
cams_dir = '/data/cams/world'


def get_cams_file(year, month, day, cams_dir=cams_dir):
    cams_file = opj(cams_dir, f'cams_forecast_{year}-{month}.nc')
    if not os.path.exists(cams_file):
        cams_file = opj(cams_dir, year, month, day,
                        f'{year}-{month}-{day}-cams-global-atmospheric-composition-forecasts.nc')
    return cams_file

## EnMAP

Each EnMAP L1C product is a directory named `ENMAP01-____L1C-DT<datatake>_<YYYYMMDDTHHMMSSZ>_...`;
the acquisition date is the sixth field of the name.

In [3]:
idir = '/data/satellite/enmap/EnMAP_L1C'
odir = '/data/satellite/enmap/L2A'
sites = ['Trasimeno']
options = '--no_clobber'   # skip the images already processed

commands = []
for site in sites:
    l1c_dir = opj(idir, site)
    if not os.path.exists(l1c_dir):
        continue
    for file in sorted(glob.glob(opj(l1c_dir, 'ENMAP*Z'))):
        date = os.path.basename(file).split('_')[5]
        cams_file = get_cams_file(date[0:4], date[4:6], date[6:8])
        commands.append(f'hgrs_enmap {file} --cams_file {cams_file} --odir {opj(odir, site)} {options}')

print('\n'.join(commands[:5]))
print(f'... {len(commands)} images')

hgrs_enmap /data/satellite/enmap/EnMAP_L1C/Trasimeno/ENMAP01-____L1C-DT0000004145_20221005T104059Z_002_V010502_20250131T022000Z --cams_file /data/cams/world/cams_forecast_2022-10.nc --odir /data/satellite/enmap/L2A/Trasimeno --no_clobber
hgrs_enmap /data/satellite/enmap/EnMAP_L1C/Trasimeno/ENMAP01-____L1C-DT0000082532_20240706T103357Z_002_V010502_20250128T220612Z --cams_file /data/cams/world/cams_forecast_2024-07.nc --odir /data/satellite/enmap/L2A/Trasimeno --no_clobber
hgrs_enmap /data/satellite/enmap/EnMAP_L1C/Trasimeno/ENMAP01-____L1C-DT0000088121_20240817T102235Z_002_V010502_20250128T220420Z --cams_file /data/cams/world/cams_forecast_2024-08.nc --odir /data/satellite/enmap/L2A/Trasimeno --no_clobber
hgrs_enmap /data/satellite/enmap/EnMAP_L1C/Trasimeno/ENMAP01-____L1C-DT0000101768_20241106T102231Z_002_V010502_20250128T215729Z --cams_file /data/cams/world/cams_forecast_2024-11.nc --odir /data/satellite/enmap/L2A/Trasimeno --no_clobber
hgrs_enmap /data/satellite/enmap/EnMAP_L1C/Trasi

## PRISMA

The PRISMA L1 file (`PRS_L1_STD_OFFL_*.he5`) is processed with the L2C file of the same acquisition
(`PRS_L2C_STD_*.he5`), from which the viewing angles are read. By default, `hgrs_prisma` looks for the
L2C file in the directory of the L1 file; otherwise give it with `--l2_path`.

In [4]:
idir = '/data/satellite/prisma/L1'
odir = '/data/satellite/prisma/L2A'
sites = ['Geneve']

commands_prisma = []
for site in sites:
    l1_dir = opj(idir, site)
    if not os.path.exists(l1_dir):
        continue
    for file in sorted(glob.glob(opj(l1_dir, 'PRS_L1_STD_OFFL_*.he5'))):
        date = os.path.basename(file).split('_')[4]
        cams_file = get_cams_file(date[0:4], date[4:6], date[6:8])
        commands_prisma.append(f'hgrs_prisma {file} --cams_file {cams_file} --odir {opj(odir, site)} {options}')

print('\n'.join(commands_prisma))

hgrs_prisma /data/satellite/prisma/L1/Geneve/PRS_L1_STD_OFFL_20200304104022_20200304104027_0001.he5 --cams_file /data/cams/world/cams_forecast_2020-03.nc --odir /data/satellite/prisma/L2A/Geneve --no_clobber
hgrs_prisma /data/satellite/prisma/L1/Geneve/PRS_L1_STD_OFFL_20200530104010_20200530104014_0001.he5 --cams_file /data/cams/world/cams_forecast_2020-05.nc --odir /data/satellite/prisma/L2A/Geneve --no_clobber
hgrs_prisma /data/satellite/prisma/L1/Geneve/PRS_L1_STD_OFFL_20210814104022_20210814104026_0001.he5 --cams_file /data/cams/world/cams_forecast_2021-08.nc --odir /data/satellite/prisma/L2A/Geneve --no_clobber
hgrs_prisma /data/satellite/prisma/L1/Geneve/PRS_L1_STD_OFFL_20210906103712_20210906103717_0001.he5 --cams_file /data/cams/world/cams_forecast_2021-09.nc --odir /data/satellite/prisma/L2A/Geneve --no_clobber


## Run the commands

Write the commands into a file and run them in parallel, for example 4 images at a time (each process
also uses several cores):

```bash
parallel -j 4 < hgrs_commands.txt
```

In [5]:
with open('hgrs_commands.txt', 'w') as f:
    f.write('\n'.join(commands + commands_prisma) + '\n')